# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### 1. What predicts health? (ML Appendix)
The paper uses Random Forest to show that the Average Position and Impressions are the top predictors of *health_score*.

**My Methodology Question:**  
The paper's methodology notes that *health_score* is directly calculated using impressions (30 pts), position (30 pts), CTR (20 pts), and scroll depth (20 pts). Because the target variable is literally constructed from these exact features, does predicting it introduce target leakage? The authors do safely note that this is "descriptive rather than causal", but a stricter validation design might predict a future health score rather than the current one to avoid a circular result.

### 2. The Anatomy of Growing Content
THe paper defines "growing vs. declining" content based on a 30-day trend (>10% increase vs. >10% decrease) and compares features like average words and age. 

**My Methodology Question:**
Where exactly in the timeline are features like *impressions* and *average_position* measure relative to the 30-day trend window? If the features are measured during the same 30-day window used to define the label, there is a risk of temporal leakage. A safer split would measure features in the prior 90 days to predict the trend of the next 30 days.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before (Random Split):** A naive random split is dangerous here because pages from the same domain (client) share branding, site architecture, and baseline CTRs. If a client's pages are in both the training and testing sets, the model just memorizes that specific client.

**After (Grouped Split):** By grouping the split by *client_id*, we force the model to train under a set of clients and test on entirely unseen clients. This is an honest test of whether the model learned universal search patterns. Below, we compare the artificially "good" random score against the honest grouped score. 

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Load data and filter to match our baseline conditions
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df_visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)].copy()

features = ['impressions_90d', 'avg_position', 'content_age_days', 'word_count', 'sessions_90d']
target = 'ctr'

df_clean = df_visible.dropna(subset=features + [target, 'client_id']).copy()
X = df_clean[features]
y = df_clean[target]
groups = df_clean['client_id']

rf = RandomForestRegressor(n_estimators=50, max_depth=7, random_state=42)

# Random split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf.fit(X_train, y_train)
random_preds = rf.predict(X_test)
random_rmse = np.sqrt(mean_squared_error(y_test, random_preds))

# Grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf.fit(X_train_grp, y_train_grp)
grp_preds = rf.predict(X_test_grp)
grp_rmse = np.sqrt(mean_squared_error(y_test_grp, grp_preds))

print(f"Dishonest Random Split RMSE (Artificially Low): {random_rmse:.4f}")
print(f"Honest Grouped Split RMSE (Realistic): {grp_rmse:.4f}")

Dishonest Random Split RMSE (Artificially Low): 0.2996
Honest Grouped Split RMSE (Realistic): 0.4198


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
X_leaked_train = X_train_grp.copy()
X_leaked_test = X_test_grp.copy()

X_leaked_train['leak_clicks_90d'] = df_clean.iloc[train_idx]['clicks_90d']
X_leaked_test['leak_clicks_90d'] = df_clean.iloc[test_idx]['clicks_90d']

rf.fit(X_leaked_train, y_train_grp)
leaked_preds = rf.predict(X_leaked_test)
leaked_rmse = np.sqrt(mean_squared_error(y_test_grp, leaked_preds))

print(f"Current Honest RMSE (Safe Features): {grp_rmse:.4f}")
print(f"Leaked Model RMSE (If we cheat): {leaked_rmse:.4f}")
print("\nAudit Verdict: Our current honest features do not contain raw clicks or product flags. The model is safe from circular leakage.")

Current Honest RMSE (Safe Features): 0.4198
Leaked Model RMSE (If we cheat): 0.2053

Audit Verdict: Our current honest features do not contain raw clicks or product flags. The model is safe from circular leakage.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Claim:** Our machine learning model accurately predicts the exact CTR a page will get, proving that refreshing our recommended pages will guarantee massive traffic growth.  
**Rewritten Claim:** Our model provides decision-support by evaluating observable search signals to identify pages that historically under-capture clicks relative to their position tier. This highlights directional opportunities for content review, prioritizing pages where a refresh is most likely to yield positive engagements. 

## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.